# Suite PYDEF — Python definitions

The names a Python module binds, in the scopes Python gives them (`Python.Definitions.define`): one entity per name
per scope, with every node that binds it, and lookup outward through enclosing functions to the module, past class
bodies. Each case checks the entities, their declarations, and what names resolve to.

In [ ]:
import { readFileSync } from "node:fs";

import { Definitions as FD, Syntax as F } from "@mbse/programs/Framework";
import { Definitions as D, Python314, Syntax as S } from "@mbse/programs/Python";
import { PythonStandard } from "@mbse/programs/Python/_Standard";

import { assert, equal } from "./support.js";

const PY315 = new PythonStandard(3, 15);

function define(text: string): [S.Module, FD.Program] {
  const module = PY315.parse(text);
  return [module, D.define(module)];
}

const listing = (program: FD.Program) =>
  [...program.entities()].map((e) => [e.kind, e.qualified_name(), e.declarations.length]);
/** The `Name` nodes of `module` spelled `spelling`, in source order. */
const names = (module: S.Module, spelling: string): S.Name[] =>
  [...F.walk(module)].filter((n): n is S.Name => n instanceof S.Name && (n.id as S.Identifier).spelling === spelling);
const refers = (program: FD.Program, node: F.Node) => D.referents(program, node).map((e) => e.qualified_name());
const one = <T>(list: readonly T[]): T => list[0] as T;

## PYDEF-01 · Names bound in a scope are local to it, one entity per name, whatever binds it

In [ ]:
{
  const [module, program] = define(`
x = 1
x += 2
y: int
a, (b, *c) = [d] = e = f()
for i in range(3):
    del i
with open() as (g, h.attribute), open() as k[0]:
    pass
try:
    pass
except E as err:
    pass
import os.path, sys as system
from . import mod as alias, other
from m import *
def function(): pass
class Klass: pass
`);
  equal(listing(program), [
    ["variable", "x", 2], ["variable", "y", 1], ["variable", "a", 1], ["variable", "b", 1], ["variable", "c", 1],
    ["variable", "d", 1], ["variable", "e", 1], ["variable", "i", 2], ["variable", "g", 1], ["variable", "err", 1],
    ["import", "os", 1], ["import", "system", 1], ["import", "alias", 1], ["import", "other", 1],
    ["function", "function", 1], ["class", "Klass", 1]]);
  const x = one(program.lookup("x"));
  assert(x.definition === names(module, "x")[0]);
  equal(x.declarations, names(module, "x"));
  assert(program.entity_of(names(module, "x")[1]) === x && program.root.kind === "module" && program.root.separator === ".");
  assert(one(program.lookup("function")).definition === module.body.at(-2));
  assert(one(program.lookup("Klass")).scope?.kind === "class");
  assert(program.scope_of(module) === program.root && program.lookup("nothing").length === 0);
  equal(refers(program, one(names(module, "f"))), []);
  equal(refers(program, one(names(module, "h"))), []); // not bound here
  console.log(listing(program).length, "entities");
}

## PYDEF-02 · Lookup goes outward through functions to the module, past class bodies; `global` and `nonlocal`

In [ ]:
{
  const [module, program] = define(`
counter = 0
nonlocal stray
class Outer:
    size = 1
    def method(self):
        return size, counter
    class Inner:
        value = size
    squares = [size for _ in range(size)]
def outer():
    global counter, created
    counter = 1
    created = 2
    def inner():
        nonlocal later
        later = 3
        def innermost():
            nonlocal later
            return later
    def unbound():
        nonlocal missing
        missing = 4
    later = 0
def reader():
    global ghost
    return ghost
`);
  const size = names(module, "size");
  equal(refers(program, size[1] as S.Name), []); // a method does not see its class's names
  equal(refers(program, names(module, "counter")[1] as S.Name), ["counter"]);
  equal(refers(program, size[2] as S.Name), []); // nor does a nested class
  equal(refers(program, size[3] as S.Name), []);
  equal(refers(program, size[4] as S.Name), ["Outer.size"]);
  equal([...program.entities()].filter((e) => ["counter", "created", "later", "missing"].includes(e.name as string))
    .map((e) => e.qualified_name()), ["counter", "created", "outer.later", "outer.unbound.missing"]);
  const later = one(program.lookup("outer.later"));
  assert(one(program.lookup("counter")).declarations.length === 2);
  equal(later.declarations, [names(module, "later")[0], names(module, "later")[2]]);
  assert(later.definition === names(module, "later")[0]); // in source order: `later = 3` in inner, then `later = 0`
  equal(refers(program, names(module, "later")[1] as S.Name), ["outer.later"]);
  const ghost = one(program.lookup("ghost"));
  assert(ghost.declarations.length === 0 && ghost.definition === null);
  equal(refers(program, one(names(module, "ghost"))), ["ghost"]);
  const outer = module.body[3] as S.FunctionDef;
  equal(refers(program, one((outer.body[0] as S.Global).names)), ["counter"]); // the names of `global` itself
  assert(one(program.lookup("stray")).parent === program.root); // `nonlocal` outside a function finds none: its own scope
  assert(program.lookup("Outer.Inner.value").length > 0 && one(program.lookup("Outer.method")).scope?.parent === program.root);
  console.log("ok");
}

## PYDEF-03 · Lambdas, comprehensions and assignment expressions

In [ ]:
{
  const [module, program] = define(`
def f(items):
    squares = [y := x * x for x in items if (last := x)]
    nested = {k: v for k, row in items for v in row}
    total = sum(n for n in items)
    pick = lambda item, key=items: (found := item)
    return y, last, x
x = 0
`);
  equal(listing(program).slice(0, 4), [["function", "f", 1], ["parameter", "f.items", 1], ["variable", "x", 1],
    ["variable", "x", 1]]); // the module's names first, then the function's
  const scopes = [...F.walk(module)].filter((c) => c instanceof S.Comprehension).map((c) => program.scope_of(c) as FD.Scope);
  equal(scopes.map((s) => s.kind), ["comprehension", "comprehension", "comprehension", "comprehension"]);
  assert(scopes[0]?.parent === one(program.lookup("f")).scope && scopes[1] === scopes[2]);
  equal(refers(program, names(module, "y")[1] as S.Name), ["f.y"]);
  equal(refers(program, names(module, "last")[1] as S.Name), ["f.last"]);
  equal(refers(program, names(module, "x").at(-2) as S.Name), ["x"]); // the comprehension's own `x` stays in it
  equal(refers(program, names(module, "items")[1] as S.Name), ["f.items"]); // the first iterable is evaluated outside
  equal([...program.entities()].filter((e) => ["item", "key", "found"].includes(e.name as string)).map((e) => e.kind),
    ["parameter", "parameter", "variable"]);
  const lam = [...F.walk(module)].find((n): n is S.Lambda => n instanceof S.Lambda) as S.Lambda;
  const key = (lam.args as S.Arguments).args[1] as S.Arg;
  assert(program.scope_of(key.default_value) === one(program.lookup("f")).scope); // defaults outside
  assert(program.scope_of(lam.body)?.kind === "lambda");
  equal(refers(program, one(names(module, "found"))), ["found"]);
  console.log("ok");
}

## PYDEF-04 · Parameters, decorators, defaults, annotations, and type parameters' annotation scopes

In [ ]:
{
  const [module, program] = define(`
T = int
@decorator(T)
def generic[T: bound, *Ts, **P = default](a: T, /, b: T = T, *c: *Ts, d, **e: P) -> T:
    return a
class Box[T](Base[T], metaclass=Meta):
    item: T
type Alias[K] = dict[K, T]
type Plain = T
`);
  const fn = one(program.lookup("generic"));
  const parameters = [...program.entities()].filter((e) => e.parent === fn.scope);
  equal(parameters.map((e) => [e.kind, e.name]), [..."abcde"].map((n) => ["parameter", n]));
  const annotations = fn.scope?.parent as FD.Scope;
  assert(annotations.kind === "type parameters");
  equal(annotations.entities().map((e) => e.name), ["T", "Ts", "P"]);
  assert(annotations.parent === program.root && annotations.node === module.body[1]);
  equal(names(module, "T").map((use) => D.referents(program, use).map((e) => e.kind)), [
    ["variable"], ["variable"], // `T = int`, and the decorator's: the module's
    ["type parameter"], ["type parameter"], ["variable"], // the annotations' are the parameter, a default the module's
    ["type parameter"], ["type parameter"], ["type parameter"], // the return annotation's, a base's, and in the class body
    ["variable"], ["variable"]]); // the type aliases', whose own parameters are others
  assert(one(program.lookup("Box")).scope?.parent?.kind === "type parameters");
  equal(refers(program, one(names(module, "Meta"))), []);
  const alias = one(program.lookup("Alias"));
  const statement = module.body[3] as S.TypeAlias;
  assert(alias.kind === "type alias" && alias.definition === statement && program.entity_of(statement.name) === alias);
  console.log("ok");
}

## PYDEF-05 · Pattern captures, and names a pattern only reads

In [ ]:
{
  const [module, program] = define(`
match command:
    case [first, *rest] | {"k": first, **rest}:
        pass
    case Point(x=0, y=py) as point if point:
        pass
    case constants.RED | _:
        pass
    case {"k": key}:
        pass
`);
  equal(listing(program), [["variable", "first", 2], ["variable", "rest", 2], ["variable", "py", 1],
    ["variable", "point", 1], ["variable", "key", 1]]);
  equal(refers(program, one(names(module, "point"))), ["point"]);
  equal(refers(program, one(names(module, "constants"))), []);
  const klass = [...F.walk(module)].find((n): n is S.MatchClass => n instanceof S.MatchClass) as S.MatchClass;
  equal(D.referents(program, one(klass.kwd_attrs)), []); // a keyword names an attribute
  console.log("ok");
}

## PYDEF-06 · What names refer to: attributes, keywords and modules' names find nothing, and lookup takes dotted names

In [ ]:
{
  const [module, program] = define(`
import a.b
class C:
    def m(self):
        return self.m(key=1)
`);
  const attribute = [...F.walk(module)].find((n): n is S.Attribute => n instanceof S.Attribute) as S.Attribute;
  const keyword = [...F.walk(module)].find((n): n is S.Keyword => n instanceof S.Keyword) as S.Keyword;
  equal(D.referents(program, attribute.attr as S.Identifier), []);
  equal(D.referents(program, keyword.arg as S.Identifier), []);
  equal(refers(program, attribute.value as S.Name), ["C.m.self"]);
  equal(refers(program, (attribute.value as S.Name).id as S.Identifier), ["C.m.self"]);
  const dotted = (one((module.body[0] as S.Import).names)).name as S.DottedName;
  equal(refers(program, dotted.names[0] as S.Identifier), ["a"]);
  equal(D.referents(program, dotted.names[1] as S.Identifier), []);
  assert(one(program.lookup("C.m.self")).kind === "parameter" && program.lookup("C.x").length === 0);
  equal(program.lookup("self", attribute), program.lookup("C.m.self"));
  assert(String(one(program.lookup("C.m"))) === "<function C.m>");
  equal(D.referents(program, new S.Identifier({ spelling: "C" })), []); // not in the program
  console.log("ok");
}

## PYDEF-07 · Every entity of the conformance source, and every name located

In [ ]:
{
  const module = Python314.parse(readFileSync("../../conformance/sources/python314.py", "utf8"));
  const program = D.define(module);
  const entities = listing(program);
  equal(entities.slice(0, 6), [["import", "annotations", 1], ["import", "osp", 1], ["import", "sys", 1],
    ["import", "sibling", 1], ["import", "one", 1], ["import", "second", 1]]);
  const has = (entry: unknown[]) => entities.some((e) => JSON.stringify(e) === JSON.stringify(entry));
  assert(has(["function", "Shape.__init__", 1]) && has(["variable", "statements.counter", 3]));
  assert(has(["type alias", "Pair", 1]) && has(["variable", "patterns.origin", 1]));
  assert([...F.walk(module)].filter((n) => n instanceof S.Name).every((n) => program.scope_of(n) !== null));
  console.log(entities.length, "entities");
}